<a href="https://colab.research.google.com/github/GGWILson-ops/MachineLearning/blob/main/Weather_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import requests #This library helps us to fetch  data from API
import pandas as pd #for handling and analysing data
import numpy as np #for numerical operation
from sklearn.model_selection import train_test_split #to split data to train and test set
from sklearn.preprocessing import LabelEncoder #to convert catogerical data into numerical values
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor #model for classification and regression task
from sklearn.metrics import mean_squared_error #meassure the accuracy of the prediction
from datetime import datetime, timedelta # handle date and time
import seaborn as sns
import matplotlib.pyplot as plt
import pytz

In [13]:
API_KEY = '7f61b6ce3af02d4a050e25dfe5169fee'
BASE_URL = 'https://api.openweathermap.org/data/2.5/'

**1.Fetch Current Weather Data**

In [14]:
def get_current_weather(city):
  url=f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"
  response=requests.get(url)
  data=response.json()
  return{
      'city':data['name'],
      'current_temp':round(data['main']['temp']),
      'feels_like':round(data['main']['feels_like']),
      'temp_min':round(data['main']['temp_min']),
      'temp_max':round(data['main']['temp_max']),
      'humidity':round(data['main']['humidity']),
      'description':data['weather'][0]['description'],
      'country':data['sys']['country'],
      'wind_gust_dir':data['wind']['deg'],
      'pressure':data['main']['pressure'],
      'Wind_Gust_Speed':data['wind']['speed']
  }

**2.Read Historical Data**

In [15]:
def read_historical_data(filename):
  df=pd.read_csv(filename)
  df=df.dropna()
  df=df.drop_duplicates()
  return df

**3.Prepare Data For Training**

In [16]:
def prepare_data(data):
  le =LabelEncoder()
  data['WindGustDir']=le.fit_transform(data['WindGustDir'])
  data['RainTomorrow']=le.fit_transform(data['RainTomorrow'])

  #define the feature variable and target variables
  x=data[['MinTemp','MaxTemp','WindGustDir','WindGustSpeed','Humidity','Pressure','Temp']] #feature variable
  y=data['RainTomorrow']#target variable

  return x,y,le #return frature, target variable and labelencoder


**4.train Rain Prediction Model**

In [17]:
def train_rain_model(x,y):
  x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
  model= RandomForestClassifier(n_estimators=100,random_state=42)
  model.fit(x_train,y_train)

  y_pred=model.predict(x_test)
  print('Mean Squared Error for Rain Model')
  print(mean_squared_error(y_test,y_pred))
  return model

**5.Prepare Regression Data**

In [18]:
def prepare_regression_data(data,feature):
  x,y=[],[]
  for i in range(len(data)-1):
    x.append(data[feature].iloc[i])
    y.append(data[feature].iloc[i+1])
  x=np.array(x).reshape(-1,1)
  y=np.array(y)
  return x,y

**Train Regression Model**

In [19]:
def train_regression_model(x,y):
  model=RandomForestRegressor(n_estimators=100, random_state=42)
  model.fit(x,y)
  return model

**Predict Future**

In [20]:
def predict_future(model,current_value):
  predictions=[current_value]

  for i in range(5):
    next_value= model.predict(np.array([[predictions[-1]]]))
    predictions.append(next_value[0])
  return predictions[1:]

**Weather Analysis Function**

In [ ]:
def weather_view():
    city = input('enter any city name: ')
    current_weather = get_current_weather(city)

    historical_data = read_historical_data('/content/drive/MyDrive/weather.csv')

    x, y, le = prepare_data(historical_data)
    rain_model = train_rain_model(x, y)

    wind_deg = current_weather['wind_gust_dir'] % 360

    compass_point = [
        ("N", 0, 11.25), ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25),
        ("ENE", 56.25, 78.75), ("E", 78.75, 101.25), ("ESE", 101.25, 123.75),
        ("SE", 123.75, 146.25), ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
        ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25), ("WSW", 236.25, 258.75),
        ("W", 258.75, 281.25), ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
        ("NNW", 326.25, 348.75)
    ]


    compass_direction = "N"
    for point, start, end in compass_point:
        if start <= wind_deg < end:
            compass_direction = point
            break

    compass_direction_encode = (
        le.transform([compass_direction])[0]
        if compass_direction in le.classes_ else -1
    )

    current_data = {
        'MinTemp': current_weather['temp_min'],
        'MaxTemp': current_weather['temp_max'],
        'WindGustDir': compass_direction_encode,
        'WindGustSpeed': current_weather['Wind_Gust_Speed'],
        'Humidity': current_weather['humidity'],
        'Pressure': current_weather['pressure'],
        'Temp': current_weather['current_temp']
    }

    current_df = pd.DataFrame([current_data])
    rain_prediction = rain_model.predict(current_df)[0]

    x_temp, y_temp = prepare_regression_data(historical_data, 'Temp')
    x_hum, y_hum = prepare_regression_data(historical_data, 'Humidity')

    temp_model = train_regression_model(x_temp, y_temp)
    hum_model = train_regression_model(x_hum, y_hum)

    future_temp = predict_future(temp_model, current_weather['temp_min'])
    future_humidity = predict_future(hum_model, current_weather['humidity'])

    timezone = pytz.timezone('Asia/Kolkata')
    now = datetime.now(timezone)

    next_hour = now + timedelta(hours=1)
    next_hour = next_hour.replace(minute=0, second=0, microsecond=0)

    future_time = [
        (next_hour + timedelta(hours=i)).strftime('%H:00')
        for i in range(5)
    ]


    print(f"City: {city}, {current_weather['country']}")
    print(f"Current Temperature: {current_weather['current_temp']}")
    print(f"Feels Like: {current_weather['feels_like']}")
    print(f"Minimum Temperature: {current_weather['temp_min']}°C")
    print(f"Maximum Temperature: {current_weather['temp_max']}°C")
    print(f"Humidity: {current_weather['humidity']}%")
    print(f"Weather Prediction: {current_weather['description']}")
    print(f"Rain Prediction: {'Yes' if rain_prediction else 'No'}")

    print('\nFuture Temperature Predictions:')
    for time, temp in zip(future_time, future_temp):
        print(f"{time}: {round(temp, 1)}°C")

    print('\nFuture Humidity Predictions:')
    for time, hum in zip(future_time, future_humidity):
        print(f"{time}: {round(hum, 1)}%")


weather_view()